# Milestone 1 - Notebook 1: Insurance Claims Fraud (Classification)
Put `insurance_claims.csv` in the same folder as this notebook, then run each cell from top to bottom.

## Step 1: Load the data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("insurance_claims.csv")
print(df.shape)
df.head()

## Step 2: Clean the data
- `_c39` is an empty column left over from the file export, so we drop it.
- Three columns use `?` to mean "unknown", and `authorities_contacted` has blanks. We label both "Unknown" so they are treated as their own category.
- ID-style columns (policy number, zip, exact location) do not help predict fraud, so we drop them.
- We turn the target `fraud_reported` into 1 (fraud) and 0 (not fraud).

In [ ]:
df = df.drop(columns=["_c39"])
df = df.replace("?", "Unknown")
df = df.fillna("Unknown")   # authorities_contacted has some truly blank entries

df["fraud"] = (df["fraud_reported"] == "Y").astype(int)

drop_cols = ["fraud_reported", "policy_number", "insured_zip",
             "incident_location", "policy_bind_date", "incident_date"]
df = df.drop(columns=drop_cols)

print(df.shape)
print("Missing values:", df.isna().sum().sum())

## Step 3: Explore the data (EDA)
This feeds your Part 5 paragraph. Look for: how rare is fraud, which numbers are related to each other, and which categories differ for fraud vs. not fraud.

In [ ]:
# How common is fraud?
print(df["fraud"].value_counts())
print(df["fraud"].value_counts(normalize=True).round(3))

# Always guessing "no fraud" would already be right this often:
print("Accuracy of always guessing no fraud:", round(1 - df["fraud"].mean(), 3))

In [ ]:
# Which numeric columns move together?
num_cols = df.select_dtypes(include="number").drop(columns=["fraud"]).columns
corr = df[num_cols].corr()

plt.figure(figsize=(10, 8))
plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar()
plt.xticks(range(len(num_cols)), num_cols, rotation=90)
plt.yticks(range(len(num_cols)), num_cols)
plt.title("Correlation of numeric columns")
plt.tight_layout()
plt.show()

In [ ]:
# Fraud rate by category (shows which groups look riskier)
for col in ["incident_severity", "insured_hobbies", "collision_type"]:
    print(df.groupby(col)["fraud"].mean().sort_values(ascending=False).round(2))
    print()

## Step 4: Prepare for modeling
Models need numbers, so we turn text categories into 0/1 columns (one-hot encoding). We keep 20% of the data aside as a test set. `stratify` keeps the fraud share the same in both parts.

In [ ]:
from sklearn.model_selection import train_test_split

X = pd.get_dummies(df.drop(columns=["fraud"]), drop_first=True)
y = df["fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Fraud share in train:", round(y_train.mean(), 3), " in test:", round(y_test.mean(), 3))

## Step 5: Baseline to beat
Every later model gets compared to this one. We report F1 and AUC (not just accuracy) because fraud is the minority class.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

base = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
pred = base.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, pred), 3))
print("F1:", round(f1_score(y_test, pred), 3))
print("AUC:", round(roc_auc_score(y_test, base.predict_proba(X_test)[:, 1]), 3))